In [2]:
from Value import Value
from Neuron import Neuron
from Layer import Layer
from NN import NN
from graph import draw_graph
from training import compute_loss
import numpy as np

In [ ]:
# Demo stuff
a = Value(3, label = "a")
b = Value(1, label = "b")
c = a * b; c.label = "c"
d = Value(-2, label = "d")
e = Value(3, label = "e")
f = d * e; f.label = "f"
g = c + f; g.label = "g"
o = g.tanh(); o.label = "o"

o.backward()

draw_graph(o)

In [ ]:
# MNIST classifier
from tensorflow.keras.datasets import mnist

# Y is the labels, x is the images
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Number of images to train on
n_train = 100
# Number of training loops
epochs = 5

x_train = x_train[:n_train]
y_train = y_train[:n_train]
dataset = x_train.reshape(100, -1) / 255.0
# Convert to Value objects
dataset = [[Value(pixel) for pixel in image] for image in dataset]
labels = y_train

# 3 layers with 16 neurons each
mlp = NN(784, [16, 16, 16, 10])
# Learning rate for SGD
lr = .01

In [ ]:
# Training loop
for i in range(epochs):
    for j in range(n_train):
        # Zero gradients
        for param in mlp.get_params():
            param.grad = 0

        # Forward pass
        logits = mlp(dataset[j])

        # Normalize logits to probabilities
        denom = sum(logit.exp() for logit in logits)

        probs = [logit.exp() / denom for logit in logits]

        for k, prob in enumerate(probs):
            prob._op = "softmax"
            prob.label = f"prob_{k}"

        # Get loss
        loss = compute_loss(probs, labels[j])

        # Backward pass
        loss.backward()
        mlp.grad_descent(lr)


In [ ]:
draw_graph(loss)